# Eksperimen SML - Ahmad Raihan
## Deteksi Berita Hoax Berbahasa Indonesia

Dataset: **IDNHoaxCorpus** (https://github.com/9uz/IDNHoaxCorpus) — 4617 tweet
berbahasa Indonesia berlabel hoax/valid/tidak terverifikasi, dikumpulkan tim
Universitas Muhammadiyah Ponorogo (2020), lisensi MIT.

Notebook ini mencakup:
1. Loading & pemeriksaan dataset mentah
2. Preprocessing teks
3. Exploratory Data Analysis (EDA)
4. Eksperimen perbandingan vectorizer & model dengan MLflow tracking

In [ ]:
import pandas as pd
import re
import string
import urllib.request

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

## 1. Load Dataset Mentah

In [ ]:
RAW_URL = "https://raw.githubusercontent.com/9uz/IDNHoaxCorpus/main/dataset/datasetUMPOHoax.csv"
urllib.request.urlretrieve(RAW_URL, "../namadataset_raw/hoax_raw.csv")

df_raw = pd.read_csv("../namadataset_raw/hoax_raw.csv")
print(df_raw.shape)
df_raw.head()

## 2. Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # buang URL
    text = re.sub(r"@\w+", " ", text)                # buang mention
    text = re.sub(r"#\w+", " ", text)                # buang hashtag
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Buang baris tanpa teks & label ambigu "?"
df = df_raw.dropna(subset=["tweet"]).copy()
df = df[df["label"].isin(["hoax", "valid"])].copy()
df["text"] = df["tweet"]
df["label"] = (df["label"] == "hoax").astype(int)  # 1=hoax, 0=valid
df = df[["text", "label"]].drop_duplicates(subset=["text"]).reset_index(drop=True)
df["clean_text"] = df["text"].apply(clean_text)

df.to_csv("namadataset_preprocessing/hoax_preprocessing.csv", index=False)
print(df.shape)
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print("=== Distribusi label (1=hoax, 0=valid) ===")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True))

print("\n=== Panjang teks (kata) ===")
print(df["text"].str.split().str.len().describe())

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["label"].value_counts().plot(kind="bar", ax=axes[0], title="Distribusi Label")
axes[0].set_xticklabels(["hoax", "valid"], rotation=0)
df["text"].str.split().str.len().hist(bins=30, ax=axes[1])
axes[1].set_title("Distribusi Panjang Teks (kata)")
plt.tight_layout()
plt.show()

## 4. Eksperimen dengan MLflow Tracking

Membandingkan kombinasi vectorizer (TF-IDF, CountVectorizer) dan model
(Logistic Regression, Naive Bayes, Random Forest — semua dengan
`class_weight=balanced` karena dataset imbalanced) menggunakan 5-fold CV.

> Catatan: `mlflow.set_tracking_uri(...)` diarahkan ke DagsHub. Ganti
> `<DAGSHUB_USERNAME>`, `<DAGSHUB_REPO>`, dan environment variable
> `MLFLOW_TRACKING_PASSWORD` (token DagsHub) sebelum menjalankan.

In [ ]:
import os

# --- Konfigurasi DagsHub MLflow tracking ---
DAGSHUB_USERNAME = "<DAGSHUB_USERNAME>"
DAGSHUB_REPO = "<DAGSHUB_REPO>"

os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
# os.environ["MLFLOW_TRACKING_PASSWORD"] = "<TOKEN_DAGSHUB_ANDA>"  # isi token DagsHub

mlflow.set_tracking_uri(f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow")
mlflow.set_experiment("eksperimen-hoax-detection")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

vectorizers = {
    "tfidf": TfidfVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2),
    "count": CountVectorizer(max_features=3000, ngram_range=(1, 2), min_df=2),
}
models = {
    "logreg_balanced": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "naive_bayes": MultinomialNB(),
    "random_forest_balanced": RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced"),
}

results = []
for vec_name, vectorizer in vectorizers.items():
    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)
    for model_name, model in models.items():
        with mlflow.start_run(run_name=f"{vec_name}_{model_name}"):
            cv_scores = cross_val_score(model, X_train_vec, y_train, cv=5, scoring="f1")
            model.fit(X_train_vec, y_train)
            y_pred = model.predict(X_test_vec)

            metrics = {
                "cv_f1_mean": cv_scores.mean(),
                "cv_f1_std": cv_scores.std(),
                "test_accuracy": accuracy_score(y_test, y_pred),
                "test_precision": precision_score(y_test, y_pred),
                "test_recall": recall_score(y_test, y_pred),
                "test_f1": f1_score(y_test, y_pred),
            }

            mlflow.log_param("vectorizer", vec_name)
            mlflow.log_param("model", model_name)
            mlflow.log_metrics(metrics)
            mlflow.sklearn.log_model(model, "model")

            results.append({"vectorizer": vec_name, "model": model_name, **metrics})

results_df = pd.DataFrame(results).sort_values("cv_f1_mean", ascending=False)
results_df

## Kesimpulan

Kandidat terbaik (F1 tertinggi pada CV) dipakai sebagai konfigurasi model
final di repo `Workflow-CI/MLProject/modelling.py`.

Setelah cell di atas dijalankan, buka dashboard DagsHub untuk melihat run
eksperimen tercatat — ini yang jadi bahan `screenshot_dashboard_*.png` dan
`screenshot_artifact_*.png` di folder `Membangun_model/`.